# Idea 1 — Track justification rate per release; block regressions

This notebook is the supporting analysis for **Idea 1** in [`PROPOSAL.md`](./PROPOSAL.md#idea-1--track-justification-rate-per-release-block-regressions). It also serves as the entry point to the welfare-thesis story for the corpus as a whole — twelve corpus-level numbers in one place, the single most-important chart, and the per-file dashboard that motivates the proposal.

The companion analysis tier (`11`–`16`) provides the slice-by-slice depth: sentence-register, emphasis/CAPS, register/stance, correlation/directiveness, ccVersion trends, and the rule-explanation pairing this notebook draws on most directly.

All linguistic and statistical terms are defined in [`GLOSSARY.md`](./GLOSSARY.md). The producer notebook (`00_data_pipeline.ipynb`) writes both data files this notebook reads.

Sections:

1. **Headline data block** — twelve corpus-level numbers, with source-notebook tags.
2. **Single most-important chart** — the cumulative `judgment_to_procedural_ratio` over ccVersion. The welfare-thesis trend made visible.
3. **Per-file dashboard** — file-level scatter (`mood_marker_pct` vs `just_ratio`) brushed-linked to category aggregates. Scroll the brush to see how individual files cluster.
4. **Per-category breakdowns** — positive-evaluative split (with the corrected 1.95× ratio) | modality.
5. **Findings** — the rule-density and justification-relevant headline patterns.
6. **Conclusions / Recommendations / Limitations** *(opinion, not data)* — the slice of the executive-summary commentary that pertains to this proposal.

In [ ]:
"""Setup: load YAML data + parquet artifact, derive helpers."""
import importlib
import altair as alt
import pandas as pd
import pathlib

import prompt_analysis
importlib.reload(prompt_analysis)
from prompt_analysis import (
    load_yaml, build_alt_df, version_order, category_colors,
    cumulative_by_version, welfare_evidence_table, positive_exemplar_table,
    SR_CLASS_COLORS, SENT_REGISTER_CLASSES, TABLEAU10,
)

alt.data_transformers.disable_max_rows()

pathlib.Path("figures").mkdir(exist_ok=True)

data              = load_yaml()
alt_df            = build_alt_df(data)
by_category       = data["by_category"]
corpus_block      = data["corpus"]
per_file_records  = data["files"]
cats              = list(by_category.keys())

CATEGORY_COLORS = category_colors(cats)
_cat_domain     = cats
_cat_range      = [CATEGORY_COLORS[c] for c in cats]

# Per-sentence parquet for forensic-evidence quoting (used by 22_audit_threat_framings most heavily).
parquet_path = pathlib.Path("sentences_classified.parquet")
sentences_df = pd.read_parquet(parquet_path) if parquet_path.exists() else None

n_sents = corpus_block["n_sents"]
n_versions = alt_df["ccVersion"].nunique()
print(f"loaded {len(per_file_records)} files | {n_sents:,} sentences | {n_versions} distinct ccVersions")
if sentences_df is not None:
    print(f"loaded sentences_classified.parquet | {len(sentences_df):,} rows")


## 1. Headline data

Twelve corpus-level numbers, in one place, with the source-notebook tag pointing to the analysis-tier notebook (`11`–`16`) that produces each one in its full chart context.

The headline number for this proposal is `pct_explained_para = 24.40%` — three out of four rule sentences in the corpus carry no justification keyword anywhere in their paragraph. The cumulative trend shown in §2 is the same metric tracked over ccVersion.

In [ ]:
"""Print the 12 most important corpus-level numbers."""
c = corpus_block["metrics"]
s = c["stance"]
sr = c["sentence_register"]
re_b = c["rule_explanation"]
jp = c["judgment_procedural"]
cf = c["consequence_framing"]
sc = c["socratic"]
af = c["address_form"]
ist = c["imperative_streaks"]

n_files = len(per_file_records)
n_sents = corpus_block["n_sents"]
n_vers  = alt_df["ccVersion"].nunique()
mood_pct = c["mood"]["marker_pct"]
imp_sent_pct = sr["imperative_sent_pct"]
appr_n   = sr["appreciative_sent_count"]
appr_pct = sr["appreciative_sent_pct"]
apol_n   = sc["apology_count"]
expl_para = re_b["pct_explained_para"]
unexpl   = re_b["pct_paragraphs_with_rules_unexplained"]
jp_ratio = jp["judgment_to_procedural_ratio"]
proc_x   = 1.0 / jp_ratio
threat_share = cf["threat_share"]
threat_n = cf["threat_count"]
causal_n = cf["causal_count"]
qual_n   = s["positive_evaluative_quality_count"]
emph_n   = s["positive_evaluative_emphasis_count"]
neg_n    = s["negative_evaluative_count"]
qual_neg = qual_n / neg_n
union_neg = (qual_n + emph_n) / neg_n
anthro_pct = af["pct_anthropomorphic"] * 100
streak_max = ist["streak_max"]

rows = [
    ("Corpus size",                      f"{n_files} files / {n_sents:,} sentences / {n_vers} ccVersions",            "00"),
    ("Imperative-marker density",        f"{mood_pct:.2f}% of word tokens",                                            "11"),
    ("Imperative sentences (%)",         f"{imp_sent_pct:.2f}% of sentences",                                          "11"),
    ("Appreciative sentences (corpus)",  f"{appr_n} / {n_sents:,} ({appr_pct:.3f}%)",                                  "11"),
    ("Apology markers (corpus)",         f"{apol_n} in {n_files} files",                                               "16"),
    ("pct_explained_para (Tier-1 headline)", f"{expl_para:.2f}% of rule sentences",                                    "16"),
    ("pct_paragraphs_with_rules_unexplained", f"{unexpl:.2f}% of rule paragraphs",                                     "16"),
    ("judgment_to_procedural_ratio",     f"{jp_ratio:.3f} (procedural is {proc_x:.1f}× more frequent)",                "16"),
    ("threat_share of explanations",     f"{threat_share:.3f} ({threat_n} threat / {causal_n} causal)",                "16"),
    ("Positive-vs-negative ratio (corrected)",
                                         f"{qual_neg:.2f}× quality-only; union ratio {union_neg:.2f}×",                "13"),
    ("pct_anthropomorphic (Claude vs the model)", f"{anthro_pct:.1f}% of named refs",                                   "16"),
    ("Longest imperative streak",        f"{streak_max} sentences in one file",                                        "16"),
]

print(f"{'Metric':<48s}  {'Value':<55s}  Source")
print("-" * 120)
for label, value, src in rows:
    print(f"{label:<48s}  {value:<55s}  {src}")


## 2. Single most-important chart — cumulative `judgment_to_procedural_ratio` over ccVersion

If you read only one chart from this analysis, read this one. The line is the running mean of `judgment_to_procedural_ratio` across every file with `ccVersion ≤ V` — so the rightmost value equals the corpus-wide per-file mean. The story it tells: early Claude Code releases peaked at ~0.65 (judgment-inviting language was a quarter as common as procedural cues), then **monotonically declined to ~0.16 at the latest version** — the corpus has gotten *less* reasoning-inviting as it has grown.

This is the welfare-thesis trend made visible. The same chart appears in `16_rule_explanation.ipynb`; re-rendered here so this proposal notebook is standalone.

In [ ]:
"""Cumulative judgment-to-procedural ratio over ccVersion (the headline trend chart)."""

df_jp_cum = alt_df[(alt_df["ccVersion"] != "")
                   & alt_df["judgment_to_procedural_ratio"].notna()]

cum_jp = cumulative_by_version(df_jp_cum, ["judgment_to_procedural_ratio"], agg="mean")

ver_order_cum = (
    alt_df[alt_df["ccVersion"] != ""]
    .drop_duplicates("ccVersion").sort_values("ccVersion_sort")["ccVersion"].tolist()
)

cum_jp_chart = (
    alt.Chart(cum_jp)
    .mark_line(point=alt.OverlayMarkDef(filled=True, size=40),
               strokeWidth=2.5, color="#4e79a7")
    .encode(
        x=alt.X("ccVersion:N", sort=ver_order_cum,
                title="ccVersion (oldest → newest)",
                axis=alt.Axis(labelAngle=-90, labelLimit=80, labelOverlap=False)),
        y=alt.Y("value:Q",
                title="judgment-to-procedural ratio (cumulative running mean)"),
        tooltip=[
            alt.Tooltip("ccVersion:N"),
            alt.Tooltip("value:Q", format=".3f", title="running mean"),
            alt.Tooltip("n_files_so_far:Q", title="files ≤ V"),
        ],
    )
    .properties(width=820, height=260,
                title="Cumulative judgment-to-procedural ratio over ccVersion (the welfare-thesis trend)")
)

cum_jp_chart.save("figures/judgment_procedural_trend.png", ppi=200)
cum_jp_chart


## 3. Per-file dashboard — imperative density vs justification ratio

Brush a region in the scatter on the left to filter the category-aggregate bars on the right. Tooltips on each point show the file's `name` and `description` from the HTML-comment frontmatter, plus the underlying densities.

The clusters that motivate Idea 1: tool-description files cluster in the upper-left (high imperative density, low justification ratio) — these are the files where the "rule without reason" pattern is most concentrated. Agent prompts and skill files cluster lower-right (lower density, higher ratio) — the positive exemplars.

In [ ]:
"""Linked dashboard: file-level scatter + brush-filtered category aggregates."""

cat_color = alt.Color("category:N",
                      scale=alt.Scale(domain=_cat_domain, range=_cat_range),
                      legend=alt.Legend(title="Category", orient="bottom", columns=4))

brush = alt.selection_interval(encodings=["x", "y"])
legend_sel = alt.selection_point(fields=["category"], bind="legend")

scatter = (
    alt.Chart(alt_df).mark_circle(opacity=0.7).encode(
        x=alt.X("mood_marker_pct:Q",
                title="Imperative-marker density (% of file tokens)"),
        y=alt.Y("just_ratio:Q", title="Justification ratio (reasons / imperative)"),
        size=alt.Size("n_tokens:Q",
                      title="tokens",
                      scale=alt.Scale(range=[20, 600])),
        color=cat_color,
        opacity=alt.condition(legend_sel, alt.value(0.85), alt.value(0.07)),
        tooltip=[
            alt.Tooltip("name:N",                   title="Name"),
            alt.Tooltip("description:N",            title="Description"),
            alt.Tooltip("ccVersion:N",              title="ccVersion"),
            alt.Tooltip("path:N",                   title="File"),
            alt.Tooltip("category:N"),
            alt.Tooltip("n_tokens:Q",               format=","),
            alt.Tooltip("imperative_sent_pct:Q",    title="imperative % of sents", format=".1f"),
            alt.Tooltip("mood_marker_pct:Q",        title="imp markers %",         format=".2f"),
            alt.Tooltip("just_ratio:Q",             title="justification ratio",   format=".2f"),
            alt.Tooltip("hard_prohibitions_pct:Q",  title="hard_proh %",            format=".2f"),
            alt.Tooltip("caps_imp_pct:Q",           title="CAPS imp %",             format=".2f"),
            alt.Tooltip("dominant_stance:N"),
            alt.Tooltip("dominant_register:N"),
            alt.Tooltip("dominant:N",               title="dominant sentence-register"),
        ]).add_params(brush, legend_sel).properties(width=470, height=420,
            title="Per-file: imperative density vs justification ratio (brush to filter →)")
)

metrics = [
    ("hard_prohibitions_pct", "Hard prohibitions %"),
    ("caps_imp_pct",          "CAPS imperative %"),
    ("all_caps_pct",          "ALL CAPS %"),
]

linked_bars = []
for col, title in metrics:
    bar = (
        alt.Chart(alt_df).mark_bar().encode(
            x=alt.X(f"mean({col}):Q", title=f"{title} (of file tokens)"),
            y=alt.Y("category:N", sort="-x", title=None),
            color=cat_color,
            tooltip=[
                alt.Tooltip("category:N"),
                alt.Tooltip(f"mean({col}):Q", title=f"mean {title}", format=".3f"),
                alt.Tooltip("count:Q", title="files in selection"),
            ]).transform_filter(brush).properties(width=260, height=130, title=f"{title} (mean, in selection)")
    )
    linked_bars.append(bar)

dashboard = scatter | alt.vconcat(*linked_bars)
dashboard


## 4. Per-category breakdowns: positive-evaluative split | modality

Two complementary per-category three-class breakdowns side by side. The left chart splits `positive_evaluative` into `positive_evaluative_quality` (genuinely affirmative — `good`, `recommended`, `optimal`, `safe`) and `positive_evaluative_emphasis` (emphasis-of-rule — `important`, `critical`, `essential`, `key`); the right chart shows the modality breakdown (deontic / epistemic / dynamic).

The **corrected positive-vs-negative ratio** printed below the left chart is **1.95×** quality-only — sharper than the original union 3.24× headline once the emphasis-of-rule words are subtracted out. PROPOSAL.md cites the corrected number.

In [ ]:
"""Per-category breakdowns: positive-evaluative split | modality (hconcat)."""

# --- positive-evaluative split (counts) ---
split_rows = []
for cat in cats:
    s = by_category[cat]["metrics"]["stance"]
    split_rows.append({"category": cat, "kind": "positive — quality",
                       "count": s["positive_evaluative_quality_count"]})
    split_rows.append({"category": cat, "kind": "positive — emphasis",
                       "count": s["positive_evaluative_emphasis_count"]})
    split_rows.append({"category": cat, "kind": "negative",
                       "count": s["negative_evaluative_count"]})
split_df = pd.DataFrame(split_rows)

split_chart = (
    alt.Chart(split_df).mark_bar().encode(
        y=alt.Y("category:N", sort=cats, title=None),
        x=alt.X("count:Q", title="raw matches per category"),
        color=alt.Color(
            "kind:N",
            scale=alt.Scale(
                domain=["positive — quality", "positive — emphasis", "negative"],
                range=["#4e79a7", "#f28e2c", "#e15759"]),
            legend=alt.Legend(title="evaluative class", orient="bottom")),
        yOffset="kind:N",
        tooltip=[alt.Tooltip("category:N"),
                 alt.Tooltip("kind:N"),
                 alt.Tooltip("count:Q", format=",")],
    ).properties(width=380, height=300,
                 title="Positive-evaluative split (counts per category)")
)

# Corpus-level summary printed alongside.
corpus_s = corpus_block["metrics"]["stance"]
q = corpus_s["positive_evaluative_quality_count"]
e = corpus_s["positive_evaluative_emphasis_count"]
n = corpus_s["negative_evaluative_count"]
print(f"corpus positive_evaluative_quality:  {q:>4d}")
print(f"corpus positive_evaluative_emphasis: {e:>4d}")
print(f"corpus negative_evaluative:          {n:>4d}")
print(f"  original ratio (q+e)/n:  {(q+e)/n:.2f}× (the 3.2× headline)")
print(f"  corrected ratio q/n:     {q/n:.2f}× (after subtracting emphasis-of-rule words)")

# --- modality (% of file tokens) ---
mod_long = pd.DataFrame([
    {"category": cat, "class": cls,
     "value": by_category[cat]["metrics"]["modality"][f"{cls}_pct"]}
    for cat in cats
    for cls in ("deontic", "epistemic", "dynamic")
])

mod_chart = (
    alt.Chart(mod_long).mark_bar().encode(
        y=alt.Y("category:N", sort=cats, title=None,
                axis=alt.Axis(labels=False, ticks=False)),
        x=alt.X("value:Q", title="% of file tokens"),
        color=alt.Color(
            "class:N",
            scale=alt.Scale(domain=["deontic", "epistemic", "dynamic"],
                            range=["#4e79a7", "#f28e2c", "#76b7b2"]),
            legend=alt.Legend(title="modality", orient="bottom")),
        yOffset="class:N",
        tooltip=[alt.Tooltip("category:N"),
                 alt.Tooltip("class:N"),
                 alt.Tooltip("value:Q", format=".3f", title="% of file tokens")],
    ).properties(width=380, height=300,
                 title="Modality (% of file tokens, per category)")
)

alt.hconcat(split_chart, mod_chart).resolve_scale(color="independent").properties(
    title=alt.TitleParams(
        "Per-category breakdowns — positive-evaluative split | modality",
        subtitle=["Both charts use the same category y-axis sort."],
        anchor="start",
    )
)


## 5. Findings — rule-density and justification patterns

The headline patterns relevant to this proposal (densities are **% of file tokens** unless stated otherwise; sentence-register figures are **% of sentences**).

- **The corpus is overwhelmingly directive prose, not conversation.** Across all 288 files, ~31% of *sentences* are imperative (per `metrics.sentence_register.imperative_sent_pct`), ~14% carry a directive marker, and only ~6% concern configuration. The lexicon confirms: **628** hard-prohibitions vs **0** profanity hits; **1387** second-person ("you") pronouns vs **153** first-person ("we/I") — a **9:1** ratio.

- **Tool descriptions are the most prohibition-heavy category.** They have the highest density of `hard_prohibitions` (~1.18% of tokens, ≈ 1 prohibition every 85 tokens) and the highest density of imperative markers. The most extreme outlier files are the bash-sandbox tool descriptions (`bash-sandbox-no-exceptions.md`, `bash-sandbox-evidence-operation-not-permitted.md`), which run at 9% hard-prohibitions per token — about one prohibition every 11 tokens.

- **Stance polarity is heavily positive — but the headline number needs correction.** With `evaluative` split into `positive_evaluative` and `negative_evaluative`, the corpus shows ~3× more positive than negative judgment markers in the union. The Opinion-round B split (above) reveals that ~40% of the "positive" count is emphasis-of-rule words (`important`, `critical`, `essential`, `key`) rather than genuinely affirmative tone. The corrected positive-vs-negative ratio (using only the quality subset) is **1.95×** — still positive, but a less strong claim than the union 3.24×.

- **Skills and Agent prompts lead on justification ratio** (~0.41 and 0.36). These prompts pair rules with stated reasons. Tool descriptions (0.15) and System reminders (0.10) issue rules without explanations. The single file with the highest ratio is `agent-prompt-agent-creation-architect.md` at 3.0. **This is the per-category gradient that Idea 1's regression-gating proposal targets.**

- **Modality (single-source spaCy detector) — what dominates per category:**
   - *Tool description* leads on deontic (0.50% of tokens) and dynamic (0.65%) — instructional "must / can do this" language with light epistemic content.
   - *System reminder* has the highest deontic density (0.88%) — short emphatic obligation prompts.
   - *Skill* and *Data / template* are modality-light overall — they describe rather than command.

The full per-file numbers and the lexicons used live in `prompt_linguistic_analysis.yaml`. The slice-by-slice analysis is in `11`–`16`; the rule-explanation pairing this proposal builds on most directly is in `16_rule_explanation.ipynb`.

***
### Conclusions for this proposal (Claude) — opinion, not data

> The conclusion that drives Idea 1 specifically:
>
> **The headline welfare claim is `pct_explained_para = 24.40%`.** Three out of every four rule sentences in the corpus are issued without any justification keyword anywhere in their paragraph. That's the single number Idea 1 leads with, and the cumulative trend chart in §2 shows the rate has been *declining* over Claude Code releases — from ~32% in early versions to ~22% now. The corpus is moving in the wrong direction, slowly but monotonically. A directional regression-gate (each release improves or holds the rate) would catch this without requiring an arbitrary absolute target.
---

***
### Recommendations for this proposal (Claude) — opinion, not data

> The asks Idea 1 makes of Anthropic, framed as "I'd want X" rather than as imperatives — Anthropic gets to decide:
>
> 1. **Justification-ratio targets in the prompt-revision workflow** — concretely, a CI check that fails (or warns loudly about) a release whose corpus-wide `pct_explained_para` drops below the previous release's. The corpus already has positive exemplars showing high rates are achievable: `system-prompt-worker-instructions.md` reaches 100% (7/7 rules explained at paragraph level). The gate is directional, not absolute — no arbitrary "30%" target.
>
> 2. **Cumulative judgment-to-procedural ratio as a release metric** — tracked over ccVersion in the same place Anthropic tracks model-eval scores. If a release ships with the ratio dropping, surface it. The current declining trend (0.65 → 0.16) is invisible without this tracking.
>
> 3. **A `## Rules (with reasons)` section convention** — counter-finding from the v2 RULES-section gap analysis: the corpus does *not* segregate rules into formal RULES sections; only 26 rule paragraphs (out of 1,268) live under such headings. I'd want a per-system-prompt convention where every rule is paired with its rationale in a structured section, auditable as a unit.
---

***
### Limitations for this proposal (Claude) — opinion, not data

> What this analysis can't tell us about the justification-rate metric specifically:
>
> 1. **One snapshot in time.** The corpus reflects Claude Code as of ccVersion 2.1.124. The cumulative trend chart shows the trajectory up to that point; what happens after is unknown — which is exactly why Idea 1 asks for a per-release re-run.
>
> 2. **Rule-based classifiers, not models.** Every metric is a keyword count or a parse-tree rule. This is audit-traceable, which is the right tradeoff for a welfare submission, but the cost is that we miss anything the lexicons don't catch — sarcasm, irony, indirect speech-acts, and explanation phrasings that don't use the canonical keywords. The justification-rate number is therefore a lower bound, not a precise count.
>
> 3. **English only.** The lexicons are English. International-language prompts (if any) would not be analyzed correctly.
>
> 4. **Not peer-reviewed.** No statistical significance tests, no formal hypothesis-testing framework. The trend claim (declining `pct_explained_para` and declining `judgment_to_procedural_ratio`) is descriptive and exploratory rather than statistically validated. The directional regression-gate proposal sidesteps this — it doesn't require a significance test; it just requires that the running mean not get worse.
---